# 04 — Temporal validation

This project uses synthetic data inspired by common online gaming analytics use cases. It does not contain confidential or proprietary Junglee Games data.

A random split of one month answers: "If tomorrow looks like today, how well do we rank?"
A temporal split answers the question operations actually has: "If we train through this month, what happens next month?"
Player behavior, promotions, and mix shift. A random split also lets near-duplicate rows of the same regime sit on both sides of the cut.


In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print("Project root:", ROOT)


Project root: /workspace/player-churn-prediction


In [2]:
metrics = json.loads((ROOT / "reports/model_metrics/metrics.json").read_text())
temporal = metrics["random_split"]["temporal_test_metrics"]
random_split = metrics["random_split"]["metrics"]
compare = pd.DataFrame([
    {"split": "Temporal: train April, test June", **{k: temporal[k] for k in ("accuracy","precision","recall","f1","roc_auc","pr_auc")}},
    {"split": "Random: 70/30 inside June", **{k: random_split[k] for k in ("accuracy","precision","recall","f1","roc_auc","pr_auc")}},
])
display(compare)
gap = random_split["pr_auc"] - temporal["pr_auc"]
display(Markdown(
    f"In-time PR-AUC is {random_split['pr_auc']:.3f}. "
    f"The April-to-June PR-AUC is {temporal['pr_auc']:.3f} "
    f"(difference {gap:+.3f}). "
    "The random split is the optimistic number. The temporal number is the one to put on a model card, "
    "because the desk will always score a later week than the one the model was fit on. "
    "Hyperparameters were chosen on April cross-validation and the threshold on May; "
    "the random-split row uses those same hyperparameters refit inside June, so the gap is about the split, not a second search on the holdout."
))


,split,accuracy,precision,recall,f1,roc_auc,pr_auc
0,"Temporal: train April, test June",0.6532,0.3536,0.7278,0.4760,0.7206,0.3719
1,Random: 70/30 inside June,0.6973,0.3941,0.7426,0.5150,0.7727,0.4548


In-time PR-AUC is 0.455. The April-to-June PR-AUC is 0.372 (difference +0.083). The random split is the optimistic number. The temporal number is the one to put on a model card, because the desk will always score a later week than the one the model was fit on. Hyperparameters were chosen on April cross-validation and the threshold on May; the random-split row uses those same hyperparameters refit inside June, so the gap is about the split, not a second search on the holdout.